In [3]:
import os, re, time, joblib, warnings
os.environ["LOKY_MAX_CPU_COUNT"] = str(os.cpu_count() or 4)

from functools import partial
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.stats import spearmanr

# RDKit
from rdkit import Chem, DataStructs, RDLogger
from rdkit.Chem import AllChem, Descriptors, MACCSkeys
from rdkit.Chem.Scaffolds import MurckoScaffold
from rdkit.Chem.MolStandardize import rdMolStandardize

# Turning off Deprecation Warning from RDKit
RDLogger.DisableLog('rdApp.*')
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

# Sklearn
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.model_selection import KFold, GroupKFold, GroupShuffleSplit, train_test_split, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.decomposition import PCA
from sklearn.feature_selection import SelectKBest, mutual_info_regression

# Models
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor, StackingRegressor
from sklearn.linear_model import Ridge
from xgboost import XGBRegressor
from sklearn.svm import SVR
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

In [ ]:
#1. PATHS + PARAMETERS
INPUT_CSV = r"D:/Machine Learning/Assay_ThT/Compiled_data.csv"
PREDICT_CSV = r"D:/Machine Learning/Assay_ThT/daxy_PTPyr.csv"
OUT_DIR = Path(r"D:/Machine Learning/Assay_ThT_2311/Final_Production_Model")
OUT_DIR.mkdir(parents=True, exist_ok=True)

IC50_CAP = 40000.0
CENSORED_POLICY = "cap"          # Options: "cap" (detection-limit substitution) or "exclude"
FP_BITS = 2048
FP_RADIUS = 2
USE_CHIRALITY_FP = True         
BEST_K_DESC = 50
BEST_K_FP = 50
RANDOM_SEED = 42

STRICT_BINARY_MI = True
DEDUPLICATE_TRAIN_SMILES = True
DEDUPLICATE_BY_ACHIRAL_2D = False  # Set True to collapse stereoisomers sharing identical 2D connectivity
USE_MOL_STANDARDIZE = True
APPLY_UNCHARGER = True
REPLICATE_DIFF_WARN_THRESHOLD = 1.0

# Hold-out External Validation settings (80% Train / 20% Independent External Test)
RUN_HOLDOUT_EXTERNAL_TEST = True
TEST_SPLIT_RATIO = 0.20
HOLDOUT_SPLIT_MODE = "kfold_fold"  # Options: "kfold_fold", "stratified", "random", or "scaffold"
HOLDOUT_FOLD_INDEX = 4             # 1 to 5 when HOLDOUT_SPLIT_MODE == "kfold_fold" (Fold 4 or 5 recommended)
STRATIFY_SPLIT_BINS = 5

#2. FEATURES + DATA PREPROCESSING (RDKIT + MACCS + ECFP4)
def load_data_robust(filepath):
    """Read files with any encoding."""
    for enc in ['utf-8', 'latin-1', 'cp1252', 'utf-16']:
        try:
            return pd.read_csv(filepath, encoding=enc)
        except Exception:
            continue
    try:
        return pd.read_excel(filepath)
    except Exception:
        return None

def standardize_smiles(smi, return_uncharge_flag=False):
    """Standardize chemical structures: cleanup, largest fragment, optional uncharge, canonicalize."""
    try:
        if pd.isna(smi) or str(smi).strip() == '':
            return (None, False) if return_uncharge_flag else None
        mol = Chem.MolFromSmiles(str(smi))
        if mol is None:
            return (None, False) if return_uncharge_flag else None

        uncharge_changed = False
        if USE_MOL_STANDARDIZE:
            clean_mol = rdMolStandardize.Cleanup(mol)
            parent_mol = rdMolStandardize.FragmentParent(clean_mol)
            smi_before_uncharge = Chem.MolToSmiles(parent_mol, canonical=True)
            if APPLY_UNCHARGER:
                uncharger = rdMolStandardize.Uncharger()
                mol = uncharger.uncharge(parent_mol)
                smi_after_uncharge = Chem.MolToSmiles(mol, canonical=True)
                uncharge_changed = (smi_before_uncharge != smi_after_uncharge)
            else:
                mol = parent_mol
        else:
            frags = Chem.GetMolFrags(mol, asMols=True)
            if len(frags) > 1:
                mol = sorted(frags, key=lambda m: m.GetNumAtoms(), reverse=True)[0]

        std_smi = Chem.MolToSmiles(
            mol,
            isomericSmiles=(not DEDUPLICATE_BY_ACHIRAL_2D),
            canonical=True
        )
        return (std_smi, uncharge_changed) if return_uncharge_flag else std_smi
    except Exception:
        return (None, False) if return_uncharge_flag else None

def parse_ic50_column(series, cap_limit=40000.0):
    """
    Parse raw IC50 entries, distinguishing exact quantitative measurements
    from right-censored strings (e.g., '>40000', '>=50000') and invalid values.
    """
    numeric_vals = []
    is_censored = []
    is_valid = []

    pattern = re.compile(r'^\s*([><=]+)?\s*([+-]?\d+(?:\.\d+)?(?:[eE][+-]?\d+)?)\s*$')
    for val in series:
        if pd.isna(val):
            numeric_vals.append(np.nan)
            is_censored.append(False)
            is_valid.append(False)
            continue
        s = str(val).strip().replace(',', '')
        match = pattern.match(s)
        if not match:
            numeric_vals.append(np.nan)
            is_censored.append(False)
            is_valid.append(False)
            continue
        qualifier, num_str = match.groups()
        num = float(num_str)
        if num <= 0:
            numeric_vals.append(np.nan)
            is_censored.append(False)
            is_valid.append(False)
            continue

        right_cens = (qualifier in ['>', '>=']) or (num > cap_limit)
        numeric_vals.append(num)
        is_censored.append(right_cens)
        is_valid.append(True)

    return pd.DataFrame({
        'raw_ic50_numeric': numeric_vals,
        'is_right_censored': is_censored,
        'is_valid_ic50': is_valid
    }, index=series.index)

def summarize_and_deduplicate_smiles(df, smi_col):
    """Assess replicate pIC50 variability before aggregating duplicate standardized SMILES."""
    counts = df.groupby('std_smiles')['pIC50'].agg(['count', 'median', 'std', lambda x: x.max() - x.min()])
    counts.columns = ['n_replicates', 'median_pIC50', 'std_pIC50', 'range_pIC50']
    multi = counts[counts['n_replicates'] > 1]
    n_multi = len(multi)
    n_high_var = int((multi['range_pIC50'] > REPLICATE_DIFF_WARN_THRESHOLD).sum()) if n_multi > 0 else 0

    print(f">>> Replicate structure analysis: {n_multi} unique structures have >= 2 measurements.")
    if n_multi > 0:
        print(f"    Mean replicate pIC50 range: {multi['range_pIC50'].mean():.4f} | "
              f"Structures with range > {REPLICATE_DIFF_WARN_THRESHOLD} log unit: {n_high_var}")

    df_dedup = df.groupby('std_smiles', as_index=False).agg({
        'pIC50': 'median',
        smi_col: 'first'
    })
    print(f">>> Deduplicated standardized SMILES: {len(df)} -> {len(df_dedup)} unique compounds (median pIC50).")
    return df_dedup

def get_bemis_murcko_scaffold(smi):
    """Extract Bemis-Murcko scaffold, grouping acyclic compounds consistently."""
    try:
        mol = Chem.MolFromSmiles(str(smi))
        if mol is None:
            return "__INVALID__"
        scaffold = MurckoScaffold.MurckoScaffoldSmiles(mol=mol, includeChirality=False)
        if not scaffold:
            return "__ACYCLIC__"
        return scaffold
    except Exception:
        return "__INVALID__"

def compute_features_inference(df_raw, training_desc_names=None):
    """
    Hybrid Features which include:
    - 2D RDKit Descriptors.
    - MACCS Keys (167 bits) with functional groups.
    - Morgan Fingerprints ECFP4 (2048 bits, chirality-aware if USE_CHIRALITY_FP=True).
    """
    print(f">>> Computing hybrid features (RDKit + MACCS + ECFP4, useChirality={USE_CHIRALITY_FP}) for {len(df_raw)} rows...")
    desc_dict = dict(Descriptors.descList)
    target_names = training_desc_names if training_desc_names is not None else [x[0] for x in Descriptors.descList]

    data_list = []
    valid_indices = []
    rdkit_fps = []

    mols = [Chem.MolFromSmiles(str(s)) if pd.notna(s) else None for s in df_raw['std_smiles']]

    for idx, (orig_idx, _) in enumerate(df_raw.iterrows()):
        mol = mols[idx]
        if mol is None:
            continue

        #2a. RDKit 2D Descriptors
        desc_vals = []
        for name in target_names:
            func = desc_dict.get(name)
            try:
                v = float(func(mol)) if func is not None else np.nan
                if not np.isfinite(v) or abs(v) > 1e6:
                    v = np.nan
                desc_vals.append(v)
            except Exception:
                desc_vals.append(np.nan)

        #2b. MACCS Fingerprints (167 bits)
        maccs_obj = MACCSkeys.GenMACCSKeys(mol)
        maccs_arr = np.array(maccs_obj, dtype=np.float64)

        #2c. Morgan Fingerprints (2048 bits)
        fp_obj = AllChem.GetMorganFingerprintAsBitVect(
            mol, FP_RADIUS, FP_BITS, useChirality=USE_CHIRALITY_FP
        )
        fp_arr = np.array(fp_obj, dtype=np.float64)

        data_list.append(np.concatenate([desc_vals, maccs_arr, fp_arr]))
        valid_indices.append(orig_idx)
        rdkit_fps.append(fp_obj)

    if len(data_list) == 0:
        raise ValueError("No valid chemical structures found after SMILES standardization.")

    X = np.array(data_list, dtype=np.float64)

    if training_desc_names is None:
        y_p = df_raw.loc[valid_indices, 'pIC50'].values
        return X, y_p, target_names, valid_indices, rdkit_fps
    else:
        return X, valid_indices, rdkit_fps

#3. PIPELINE, METRICS & APPLICABILITY DOMAIN
class SafeVarianceThreshold(BaseEstimator, TransformerMixin):
    """Variance filter that retains the highest-variance feature if all features fall below threshold."""
    def __init__(self, threshold=0.0):
        self.threshold = threshold

    def fit(self, X, y=None):
        X_arr = np.asarray(X)
        if X_arr.ndim != 2 or X_arr.shape[1] == 0:
            raise ValueError("Input contains no features for variance filtering.")
        variances = np.var(X_arr, axis=0)
        self.support_ = variances > self.threshold
        if not np.any(self.support_):
            warnings.warn(
                f"No features exceeded variance threshold ({self.threshold}); retaining highest-variance feature.",
                RuntimeWarning
            )
            self.support_[int(np.argmax(variances))] = True
        return self

    def transform(self, X):
        return np.asarray(X)[:, self.support_]

class AdaptiveSelectKBest(BaseEstimator, TransformerMixin):
    """SelectKBest wrapper that safely adapts k if fewer features survive variance filtering."""
    def __init__(self, score_func, k=50):
        self.score_func = score_func
        self.k = k

    def fit(self, X, y):
        if X.shape[1] == 0:
            raise ValueError("No features remaining after variance filtering.")
        k_eff = min(self.k, X.shape[1]) if isinstance(self.k, int) else self.k
        self.selector_ = SelectKBest(score_func=self.score_func, k=k_eff)
        self.selector_.fit(X, y)
        return self

    def transform(self, X):
        return self.selector_.transform(X)

def get_pipeline(n_desc_cols, model_type='xgb', k_desc=120, k_fp=350, random_state=42):
    if STRICT_BINARY_MI:
        mi_desc = partial(mutual_info_regression, discrete_features=False, random_state=random_state)
        mi_fp = partial(mutual_info_regression, discrete_features=True, random_state=random_state)
    else:
        mi_desc = partial(mutual_info_regression, random_state=random_state)
        mi_fp = partial(mutual_info_regression, random_state=random_state)

    desc_pipe = Pipeline([
        ('imp', SimpleImputer(strategy='median')),
        ('var', SafeVarianceThreshold(threshold=0.0)),
        ('scaler', RobustScaler()),
        ('selector', AdaptiveSelectKBest(score_func=mi_desc, k=k_desc))
    ])

    fp_pipe = Pipeline([
        ('var', SafeVarianceThreshold(threshold=0.01)),
        ('selector', AdaptiveSelectKBest(score_func=mi_fp, k=k_fp))
    ])

    preprocessor = ColumnTransformer(transformers=[
        ('desc_proc', desc_pipe, slice(0, n_desc_cols)),
        ('fp_proc', fp_pipe, slice(n_desc_cols, None))
    ])

    if model_type == 'xgb':
        model = XGBRegressor(
            n_estimators=500, max_depth=4, learning_rate=0.02,
            min_child_weight=2, reg_alpha=0.2, reg_lambda=1.5,
            subsample=0.8, colsample_bytree=0.7, n_jobs=1, random_state=random_state
        )
    elif model_type == 'rf':
        model = RandomForestRegressor(
            n_estimators=400, max_depth=12, min_samples_leaf=2,
            max_features=0.5, n_jobs=1, random_state=random_state
        )
    elif model_type == 'et':
        model = ExtraTreesRegressor(
            n_estimators=400, max_depth=14, min_samples_leaf=2,
            max_features=0.5, n_jobs=1, random_state=random_state
        )
    else:
        model = Pipeline([
            ('scale', StandardScaler()),
            ('svm', SVR(C=2.5, epsilon=0.05, kernel='rbf', gamma='scale'))
        ])

    return Pipeline([('preprocessor', preprocessor), ('model', model)])

def compute_qsar_metrics(y_true, y_pred):
    """
    Compute standard QSAR validation metrics including Golbraikh-Tropsha (r, k, k', r0^2, r0'^2)
    and Roy rm2 metrics alongside R2, RMSE, and MAE.
    """
    y_true = np.asarray(y_true, dtype=np.float64)
    y_pred = np.asarray(y_pred, dtype=np.float64)

    r2_val = float(r2_score(y_true, y_pred))
    rmse_val = float(np.sqrt(mean_squared_error(y_true, y_pred)))
    mae_val = float(mean_absolute_error(y_true, y_pred))

    r_corr = float(np.corrcoef(y_true, y_pred)[0, 1]) if len(y_true) > 1 else np.nan
    r2_corr = float(r_corr ** 2) if np.isfinite(r_corr) else np.nan

    denom_pred = float(np.sum(y_pred ** 2))
    denom_true = float(np.sum(y_true ** 2))
    k = float(np.sum(y_true * y_pred) / denom_pred) if denom_pred > 0 else np.nan
    k_prime = float(np.sum(y_true * y_pred) / denom_true) if denom_true > 0 else np.nan

    tss_true = float(np.sum((y_true - np.mean(y_true)) ** 2))
    tss_pred = float(np.sum((y_pred - np.mean(y_pred)) ** 2))
    r0_2 = float(1.0 - np.sum((y_true - k * y_pred) ** 2) / tss_true) if tss_true > 0 else np.nan
    r0_prime_2 = float(1.0 - np.sum((y_pred - k_prime * y_true) ** 2) / tss_pred) if tss_pred > 0 else np.nan

    rm2 = float(r2_corr * (1.0 - np.sqrt(max(0.0, r2_corr - r0_2)))) if np.isfinite(r0_2) else np.nan
    rm2_prime = float(r2_corr * (1.0 - np.sqrt(max(0.0, r2_corr - r0_prime_2)))) if np.isfinite(r0_prime_2) else np.nan
    rm2_avg = 0.5 * (rm2 + rm2_prime)

    return {
        'R2': r2_val,
        'RMSE': rmse_val,
        'MAE': mae_val,
        'r': r_corr,
        'r2_corr': r2_corr,
        'k': k,
        'k_prime': k_prime,
        'r0_2': r0_2,
        'r0_prime_2': r0_prime_2,
        'rm2_avg': rm2_avg
    }

def build_applicability_domain(X_train, y_train, train_fps, train_smiles, n_desc_cols, n_pca_ad=20):
    """
    Build Applicability Domain using continuous descriptors for PCA-leverage
    and ECFP4 fingerprints for structural Tanimoto similarity.
    """
    X_desc_train = X_train[:, :n_desc_cols]
    imp_var_scaler = Pipeline([
        ('imp', SimpleImputer(strategy='median')),
        ('var', SafeVarianceThreshold(threshold=0.0)),
        ('scaler', StandardScaler())
    ])
    X_scaled = imp_var_scaler.fit_transform(X_desc_train)
    n_comp = min(n_pca_ad, X_scaled.shape[0] - 1, X_scaled.shape[1])
    if n_comp < 1:
        raise ValueError("Insufficient samples or features for AD PCA construction.")

    ad_pipe = Pipeline([
        ('preprocess', imp_var_scaler),
        ('pca', PCA(n_components=n_comp, random_state=RANDOM_SEED))
    ])
    Z_train = ad_pipe.fit_transform(X_desc_train)
    n_samples, p_dim = Z_train.shape
    mean_z = Z_train.mean(axis=0)
    Z_train_c = Z_train - mean_z
    inv_gram = np.linalg.pinv(Z_train_c.T @ Z_train_c)
    h_train = (1.0 / n_samples) + np.sum((Z_train_c @ inv_gram) * Z_train_c, axis=1)
    h_star = 3.0 * (p_dim + 1) / n_samples

    return {
        'ad_pipe': ad_pipe,
        'n_desc_cols': n_desc_cols,
        'mean_z': mean_z,
        'inv_gram': inv_gram,
        'n_train': n_samples,
        'p_dim': p_dim,
        'h_star': h_star,
        'h_train': h_train,
        'train_fps': train_fps,
        'train_smiles': list(train_smiles),
        'train_y': y_train
    }

def evaluate_new_applicability_domain(X_new, new_fps, ad_bundle):
    n_desc_cols = ad_bundle['n_desc_cols']
    X_desc_new = X_new[:, :n_desc_cols]
    Z_new = ad_bundle['ad_pipe'].transform(X_desc_new)
    Z_new_c = Z_new - ad_bundle['mean_z']
    h_new = (1.0 / ad_bundle['n_train']) + np.sum((Z_new_c @ ad_bundle['inv_gram']) * Z_new_c, axis=1)
    h_star = ad_bundle['h_star']
    train_fps = ad_bundle['train_fps']
    train_smiles = ad_bundle['train_smiles']
    train_y = ad_bundle['train_y']

    max_tanimoto = []
    top5_tanimoto = []
    nearest_smi = []
    nearest_pic50 = []

    for fp in new_fps:
        sims = np.array(DataStructs.BulkTanimotoSimilarity(fp, train_fps))
        best_idx = int(np.argmax(sims))
        top5_vals = np.sort(sims)[-5:]

        max_tanimoto.append(float(sims[best_idx]))
        top5_tanimoto.append(float(np.mean(top5_vals)))
        nearest_smi.append(train_smiles[best_idx])
        nearest_pic50.append(float(train_y[best_idx]))

    max_tanimoto = np.array(max_tanimoto)
    top5_tanimoto = np.array(top5_tanimoto)

    ad_status = []
    for h_val, s_max in zip(h_new, max_tanimoto):
        if (h_val <= h_star) and (s_max >= 0.35):
            ad_status.append("In-domain")
        elif (h_val <= h_star * 1.5) and (s_max >= 0.25):
            ad_status.append("Borderline")
        else:
            ad_status.append("Out-of-domain")

    return {
        'Leverage_h': h_new,
        'Leverage_Threshold_h_star': np.full(len(h_new), h_star),
        'Max_Tanimoto_Train': max_tanimoto,
        'Top5_Mean_Tanimoto': top5_tanimoto,
        'Nearest_Train_SMILES': nearest_smi,
        'Nearest_Train_pIC50': nearest_pic50,
        'AD_Reliability': ad_status
    }

def evaluate_ad_error_stratification(X, y, oof_preds, train_fps, train_smiles, n_desc, splits, scheme_name="Random CV"):
    """
    Evaluate held-out prediction error (MAE, 95% CI, RMSE) stratified across predefined AD categories
    and quantify Spearman correlation between AD metrics and absolute prediction error.
    """
    oof_ad_labels = np.empty(len(y), dtype=object)
    oof_tanimoto = np.zeros(len(y), dtype=np.float64)
    oof_leverage_ratio = np.zeros(len(y), dtype=np.float64)

    for train_idx, val_idx in splits:
        fold_fps_tr = [train_fps[i] for i in train_idx]
        fold_fps_val = [train_fps[i] for i in val_idx]
        fold_smi_tr = [train_smiles[i] for i in train_idx]
        fold_ad_bundle = build_applicability_domain(
            X[train_idx], y[train_idx], fold_fps_tr, fold_smi_tr, n_desc_cols=n_desc
        )
        fold_ad_res = evaluate_new_applicability_domain(X[val_idx], fold_fps_val, fold_ad_bundle)
        oof_ad_labels[val_idx] = fold_ad_res['AD_Reliability']
        oof_tanimoto[val_idx] = fold_ad_res['Max_Tanimoto_Train']
        oof_leverage_ratio[val_idx] = fold_ad_res['Leverage_h'] / fold_ad_res['Leverage_Threshold_h_star']

    abs_err = np.abs(y - oof_preds)
    rho_sim, p_sim = spearmanr(oof_tanimoto, abs_err)
    rho_lev, p_lev = spearmanr(oof_leverage_ratio, abs_err)

    print(f"\n--- Cross-Validated AD Error Stratification ({scheme_name}) ---")
    print(f"  Spearman(Max_Tanimoto, |Error|): rho = {rho_sim:.4f} (p = {p_sim:.4g})")
    print(f"  Spearman(Leverage h/h*, |Error|): rho = {rho_lev:.4f} (p = {p_lev:.4g})")
    for label in ["In-domain", "Borderline", "Out-of-domain"]:
        mask = (oof_ad_labels == label)
        count = int(np.sum(mask))
        if count > 0:
            err_sub = abs_err[mask]
            mae_g = float(np.mean(err_sub))
            rmse_g = float(np.sqrt(mean_squared_error(y[mask], oof_preds[mask])))
            ci95 = float(1.96 * np.std(err_sub, ddof=1) / np.sqrt(count)) if count > 1 else np.nan
            ci_str = f"+/- {ci95:.4f}" if np.isfinite(ci95) else "N/A (n=1)"
            print(f"  {label:<15} | Count: {count:>4d} | MAE: {mae_g:.4f} ({ci_str}) | RMSE: {rmse_g:.4f}")
        else:
            print(f"  {label:<15} | Count:    0 | MAE: N/A | RMSE: N/A")

def run_holdout_external_validation(X, y, n_desc, train_fps, train_smiles, split_ratio=0.20, mode="kfold_fold", fold_index=4):
    """
    Partition dataset into 80% Training Set and 20% Independent Hold-Out External Test Set.
    Train the stacking workflow strictly on the 80% Training Set and report R2_train, RMSE_train,
    Q2_CV(train), R2_test(external), RMSE_test(external), r, k, k', and AD stratification on the test set.
    """
    mode_label = f"KFOLD FOLD #{fold_index}" if mode == "kfold_fold" else mode.upper()
    print("\n" + "="*68)
    print(f"  PHASE 1: 80/20 INDEPENDENT HOLD-OUT EXTERNAL VALIDATION ({mode_label})")
    print("="*68)

    all_idx = np.arange(len(y))
    if mode == "kfold_fold":
        cv_outer = KFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)
        splits_list = list(cv_outer.split(X, y))
        idx_pos = max(0, min(4, int(fold_index) - 1))
        tr_idx, te_idx = splits_list[idx_pos]
    elif mode == "scaffold":
        scaffolds = np.array([get_bemis_murcko_scaffold(s) for s in train_smiles])
        gss = GroupShuffleSplit(n_splits=1, test_size=split_ratio, random_state=RANDOM_SEED)
        tr_idx, te_idx = next(gss.split(X, y, groups=scaffolds))
    elif mode == "stratified":
        bins = pd.qcut(y, q=STRATIFY_SPLIT_BINS, labels=False, duplicates='drop')
        tr_idx, te_idx = train_test_split(all_idx, test_size=split_ratio, stratify=bins, random_state=RANDOM_SEED)
    else:
        tr_idx, te_idx = train_test_split(all_idx, test_size=split_ratio, random_state=RANDOM_SEED)

    X_tr, y_tr = X[tr_idx], y[tr_idx]
    X_te, y_te = X[te_idx], y[te_idx]
    fps_tr = [train_fps[i] for i in tr_idx]
    fps_te = [train_fps[i] for i in te_idx]
    smi_tr = [train_smiles[i] for i in tr_idx]
    smi_te = [train_smiles[i] for i in te_idx]

    print(f">>> Split summary: Train n = {len(tr_idx)} (mean pIC50 = {y_tr.mean():.3f} +/- {y_tr.std():.3f}) | "
          f"External Test n = {len(te_idx)} (mean pIC50 = {y_te.mean():.3f} +/- {y_te.std():.3f})")

    p_xgb = get_pipeline(n_desc, 'xgb', k_desc=BEST_K_DESC, k_fp=BEST_K_FP, random_state=RANDOM_SEED)
    p_rf = get_pipeline(n_desc, 'rf', k_desc=BEST_K_DESC, k_fp=BEST_K_FP, random_state=RANDOM_SEED)
    p_et = get_pipeline(n_desc, 'et', k_desc=BEST_K_DESC, k_fp=BEST_K_FP, random_state=RANDOM_SEED)
    p_svr = get_pipeline(n_desc, 'svr', k_desc=BEST_K_DESC, k_fp=BEST_K_FP, random_state=RANDOM_SEED)

    cv_inner = KFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)
    inner_splits = list(cv_inner.split(X_tr, y_tr))

    # Leak-free Nested 5-Fold CV inside the 80% Training Set
    nested_stack_tr = StackingRegressor(
        estimators=[('xgb', p_xgb), ('rf', p_rf), ('et', p_et), ('svr', p_svr)],
        final_estimator=Ridge(alpha=5.0, positive=True, random_state=RANDOM_SEED),
        cv=cv_inner,
        n_jobs=1
    )
    oof_tr_nested = cross_val_predict(nested_stack_tr, X_tr, y_tr, cv=inner_splits, n_jobs=-1)
    m_cv_tr = compute_qsar_metrics(y_tr, oof_tr_nested)

    # Fit stacking model on 80% Training Set
    o_xgb = cross_val_predict(p_xgb, X_tr, y_tr, cv=inner_splits, n_jobs=-1)
    o_rf = cross_val_predict(p_rf, X_tr, y_tr, cv=inner_splits, n_jobs=-1)
    o_et = cross_val_predict(p_et, X_tr, y_tr, cv=inner_splits, n_jobs=-1)
    o_svr = cross_val_predict(p_svr, X_tr, y_tr, cv=inner_splits, n_jobs=-1)

    X_meta_tr_oof = np.column_stack([o_xgb, o_rf, o_et, o_svr])
    meta_tr = Ridge(alpha=5.0, positive=True, random_state=RANDOM_SEED)
    meta_tr.fit(X_meta_tr_oof, y_tr)

    p_xgb.fit(X_tr, y_tr)
    p_rf.fit(X_tr, y_tr)
    p_et.fit(X_tr, y_tr)
    p_svr.fit(X_tr, y_tr)

    # Training Set (80%) Predictions
    tr_base_preds = np.column_stack([
        p_xgb.predict(X_tr), p_rf.predict(X_tr), p_et.predict(X_tr), p_svr.predict(X_tr)
    ])
    y_tr_pred = meta_tr.predict(tr_base_preds)
    m_tr = compute_qsar_metrics(y_tr, y_tr_pred)

    # Independent External Test Set (20%) Predictions
    te_p1 = p_xgb.predict(X_te)
    te_p2 = p_rf.predict(X_te)
    te_p3 = p_et.predict(X_te)
    te_p4 = p_svr.predict(X_te)
    te_base_preds = np.column_stack([te_p1, te_p2, te_p3, te_p4])
    y_te_pred = meta_tr.predict(te_base_preds)
    te_disagreement = np.std(te_base_preds, axis=1, ddof=1)
    m_te = compute_qsar_metrics(y_te, y_te_pred)

    print("\n--- Summary Table (Thesis Format: Train 80% vs. External Test 20%) ---")
    header = f"{'Model':<14} | {'R2_train':>8} | {'RMSE_train':>10} | {'Q2_CV(tr)':>9} | {'R2_test':>8} | {'RMSE_test':>9} | {'r':>7} | {'k':>7} | {'k_prime':>7}"
    print(header)
    print("-" * len(header))
    for name, tr_col, oof_col, te_col in [
        ('XGBoost', tr_base_preds[:, 0], o_xgb, te_p1),
        ('RandomForest', tr_base_preds[:, 1], o_rf, te_p2),
        ('ExtraTrees', tr_base_preds[:, 2], o_et, te_p3),
        ('SVR', tr_base_preds[:, 3], o_svr, te_p4),
        ('Stacking (ML)', y_tr_pred, oof_tr_nested, y_te_pred)
    ]:
        mt_i = compute_qsar_metrics(y_tr, tr_col)
        mc_i = compute_qsar_metrics(y_tr, oof_col)
        me_i = compute_qsar_metrics(y_te, te_col)
        print(f"{name:<14} | {mt_i['R2']:>8.4f} | {mt_i['RMSE']:>10.4f} | {mc_i['R2']:>9.4f} | "
              f"{me_i['R2']:>8.4f} | {me_i['RMSE']:>9.4f} | {me_i['r']:>7.4f} | {me_i['k']:>7.4f} | {me_i['k_prime']:>7.4f}")

    print("\n--- Golbraikh-Tropsha & Roy Criteria on 20% External Test Set (Stacking) ---")
    print(f"  R2_test = {m_te['R2']:.4f} (Threshold > 0.60) | r^2 = {m_te['r2_corr']:.4f} | MAE_test = {m_te['MAE']:.4f}")
    print(f"  k = {m_te['k']:.4f} | k' = {m_te['k_prime']:.4f} (Threshold: 0.85 <= k, k' <= 1.15)")
    print(f"  r0^2 = {m_te['r0_2']:.4f} | r0'^2 = {m_te['r0_prime_2']:.4f} | rm2_avg = {m_te['rm2_avg']:.4f}")

    # Applicability Domain evaluation on the 20% External Test Set
    ad_bundle_tr = build_applicability_domain(X_tr, y_tr, fps_tr, smi_tr, n_desc_cols=n_desc)
    ad_te_res = evaluate_new_applicability_domain(X_te, fps_te, ad_bundle_tr)
    abs_err_te = np.abs(y_te - y_te_pred)

    print("\n--- External Test Set AD Error Stratification (20% Hold-Out) ---")
    for label in ["In-domain", "Borderline", "Out-of-domain"]:
        mask = (np.array(ad_te_res['AD_Reliability']) == label)
        cnt = int(np.sum(mask))
        if cnt > 0:
            mae_g = float(np.mean(abs_err_te[mask]))
            rmse_g = float(np.sqrt(mean_squared_error(y_te[mask], y_te_pred[mask])))
            r2_g = float(r2_score(y_te[mask], y_te_pred[mask])) if cnt > 2 else np.nan
            print(f"  {label:<15} | Count: {cnt:>4d} | R2_test: {r2_g:>7.4f} | MAE_test: {mae_g:.4f} | RMSE_test: {rmse_g:.4f}")
        else:
            print(f"  {label:<15} | Count:    0 | R2_test:     N/A | MAE_test: N/A    | RMSE_test: N/A")

    # Export Hold-Out External Test Set predictions
    df_ext_out = pd.DataFrame({
        'Dataset_Index': te_idx + 1,
        'std_smiles': smi_te,
        'Experimental_pIC50': y_te,
        'Predicted_pIC50': y_te_pred,
        'Absolute_Error': abs_err_te,
        'Pred_XGB': te_p1,
        'Pred_RF': te_p2,
        'Pred_ET': te_p3,
        'Pred_SVR': te_p4,
        'BaseModel_Prediction_Disagreement': te_disagreement
    })
    for k_ad, v_ad in ad_te_res.items():
        df_ext_out[k_ad] = v_ad

    ext_csv_path = OUT_DIR / "Holdout_External_Test_Predictions_with_AD.csv"
    df_ext_out.to_csv(ext_csv_path, index=False, encoding='utf-8-sig')
    print(f">>> Saved 20% Hold-Out External Test predictions to: {ext_csv_path}")

    return {
        'train_metrics': m_tr,
        'cv_train_metrics': m_cv_tr,
        'external_test_metrics': m_te
    }

def train_final_stacking(X, y, n_desc, train_fps, train_smiles):
    print("\n" + "="*68)
    print("  PHASE 2: FULL-DATASET CROSS-VALIDATION & PRODUCTION MODEL REFIT")
    print("="*68)

    p_xgb = get_pipeline(n_desc, 'xgb', k_desc=BEST_K_DESC, k_fp=BEST_K_FP, random_state=RANDOM_SEED)
    p_rf = get_pipeline(n_desc, 'rf', k_desc=BEST_K_DESC, k_fp=BEST_K_FP, random_state=RANDOM_SEED)
    p_et = get_pipeline(n_desc, 'et', k_desc=BEST_K_DESC, k_fp=BEST_K_FP, random_state=RANDOM_SEED)
    p_svr = get_pipeline(n_desc, 'svr', k_desc=BEST_K_DESC, k_fp=BEST_K_FP, random_state=RANDOM_SEED)

    cv_random = KFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)
    random_splits = list(cv_random.split(X, y))
    print("Generating 5-fold OOF predictions for Base Models (XGB, RF, ExtraTrees, SVR)...")

    o_xgb = cross_val_predict(p_xgb, X, y, cv=random_splits, n_jobs=-1)
    o_rf = cross_val_predict(p_rf, X, y, cv=random_splits, n_jobs=-1)
    o_et = cross_val_predict(p_et, X, y, cv=random_splits, n_jobs=-1)
    o_svr = cross_val_predict(p_svr, X, y, cv=random_splits, n_jobs=-1)

    X_meta = np.column_stack([o_xgb, o_rf, o_et, o_svr])
    meta_model = Ridge(alpha=5.0, positive=True, random_state=RANDOM_SEED)

    # a. Leak-free Nested Random 5-Fold CV
    print("Evaluating leak-free Nested Random 5-Fold CV...")
    nested_stack = StackingRegressor(
        estimators=[('xgb', p_xgb), ('rf', p_rf), ('et', p_et), ('svr', p_svr)],
        final_estimator=Ridge(alpha=5.0, positive=True, random_state=RANDOM_SEED),
        cv=cv_random,
        n_jobs=1
    )
    oof_stacking = cross_val_predict(nested_stack, X, y, cv=random_splits, n_jobs=-1)

    q2_cv = r2_score(y, oof_stacking)
    rmse_cv = np.sqrt(mean_squared_error(y, oof_stacking))
    mae_cv = mean_absolute_error(y, oof_stacking)

    print(f"Base OOF R2 (Random CV) -> XGB: {r2_score(y, o_xgb):.4f} | RF: {r2_score(y, o_rf):.4f} | ET: {r2_score(y, o_et):.4f} | SVR: {r2_score(y, o_svr):.4f}")
    print(f"Stacking Nested Random 5-Fold -> Q2_CV: {q2_cv:.4f} | RMSE_CV: {rmse_cv:.4f} | MAE_CV: {mae_cv:.4f}")

    # Per-Fold Outer Test Breakdown (Folds 1 to 5)
    print("\n--- Per-Fold Outer Test Performance Breakdown (Nested Random 5-Fold CV) ---")
    f_header = f"{'Fold':<10} | {'n_test':>6} | {'R2_test':>8} | {'RMSE_test':>9} | {'MAE_test':>8} | {'r':>7} | {'k':>7} | {'k_prime':>7} | {'rm2_avg':>7}"
    print(f_header)
    print("-" * len(f_header))
    fold_metrics_list = []
    for f_idx, (_, te_i) in enumerate(random_splits, 1):
        mf = compute_qsar_metrics(y[te_i], oof_stacking[te_i])
        fold_metrics_list.append(mf)
        print(f"{'Fold ' + str(f_idx):<10} | {len(te_i):>6d} | {mf['R2']:>8.4f} | {mf['RMSE']:>9.4f} | "
              f"{mf['MAE']:>8.4f} | {mf['r']:>7.4f} | {mf['k']:>7.4f} | {mf['k_prime']:>7.4f} | {mf['rm2_avg']:>7.4f}")
    print("-" * len(f_header))
    print(f"{'Mean':<10} | {len(y)//5:>6d} | {np.mean([m['R2'] for m in fold_metrics_list]):>8.4f} | "
          f"{np.mean([m['RMSE'] for m in fold_metrics_list]):>9.4f} | {np.mean([m['MAE'] for m in fold_metrics_list]):>8.4f} | "
          f"{np.mean([m['r'] for m in fold_metrics_list]):>7.4f} | {np.mean([m['k'] for m in fold_metrics_list]):>7.4f} | "
          f"{np.mean([m['k_prime'] for m in fold_metrics_list]):>7.4f} | {np.mean([m['rm2_avg'] for m in fold_metrics_list]):>7.4f}")

    evaluate_ad_error_stratification(X, y, oof_stacking, train_fps, train_smiles, n_desc, random_splits, scheme_name="Nested Random 5-Fold CV")

    # b. Scaffold-Grouped Outer 5-Fold CV (with Random Inner Stacking CV)
    scaffolds = np.array([get_bemis_murcko_scaffold(s) for s in train_smiles])
    scaff_series = pd.Series(scaffolds).value_counts()
    unique_scaffolds = len(scaff_series)
    n_acyclic = int((scaffolds == "__ACYCLIC__").sum())
    print(f"\n>>> Bemis-Murcko Scaffold Summary: {unique_scaffolds} unique scaffold groups | Acyclic compounds: {n_acyclic}")
    print(f"    Top 5 largest scaffold group sizes: {scaff_series.head(5).to_dict()}")

    q2_scaff, rmse_scaff = np.nan, np.nan
    if unique_scaffolds >= 5:
        cv_scaffold = GroupKFold(n_splits=5)
        scaffold_splits = list(cv_scaffold.split(X, y, scaffolds))
        min_train_size = min(len(train_idx) for train_idx, _ in scaffold_splits)
        if min_train_size < 5:
            warnings.warn(
                f"Minimum outer training fold size ({min_train_size}) is < 5; skipping Scaffold-Grouped CV.",
                RuntimeWarning
            )
        else:
            print("Evaluating Scaffold-Grouped Outer 5-Fold CV (with Random Inner Stacking CV)...")
            oof_scaffold = cross_val_predict(nested_stack, X, y, cv=scaffold_splits, n_jobs=-1)
            q2_scaff = r2_score(y, oof_scaffold)
            rmse_scaff = np.sqrt(mean_squared_error(y, oof_scaffold))
            mae_scaff = mean_absolute_error(y, oof_scaffold)
            print(f"Stacking Scaffold-Grouped Outer 5-Fold -> Q2_Scaff: {q2_scaff:.4f} | RMSE_Scaff: {rmse_scaff:.4f} | MAE_Scaff: {mae_scaff:.4f}")
            evaluate_ad_error_stratification(
                X, y, oof_scaffold, train_fps, train_smiles, n_desc, scaffold_splits, scheme_name="Scaffold-Grouped Outer 5-Fold CV"
            )

    print("\nFitting Meta-Learner on full OOF matrix and Refitting Base Models on 100% dataset...")
    meta_model.fit(X_meta, y)
    print(f"Meta-Learner Weights (XGB, RF, ET, SVR): {np.round(meta_model.coef_, 4)}")

    p_xgb.fit(X, y)
    p_rf.fit(X, y)
    p_et.fit(X, y)
    p_svr.fit(X, y)

    full_tr_preds = meta_model.predict(np.column_stack([
        p_xgb.predict(X), p_rf.predict(X), p_et.predict(X), p_svr.predict(X)
    ]))
    m_full_tr = compute_qsar_metrics(y, full_tr_preds)
    print(f"Full Production Model (100% Fit) -> R2_train: {m_full_tr['R2']:.4f} | RMSE_train: {m_full_tr['RMSE']:.4f}")

    ad_bundle = build_applicability_domain(X, y, train_fps, train_smiles, n_desc_cols=n_desc)

    return {
        'xgb': p_xgb, 'rf': p_rf, 'et': p_et, 'svr': p_svr,
        'meta': meta_model,
        'n_desc': n_desc,
        'oof_preds': oof_stacking,
        'q2_cv': q2_cv,
        'rmse_cv': rmse_cv,
        'q2_scaffold_cv': q2_scaff,
        'rmse_scaffold_cv': rmse_scaff,
        'ad_bundle': ad_bundle
    }

In [ ]:
#4. MAIN WORKFLOW
def main():
    print(f">>> Loading training data: {INPUT_CSV}")
    df_train = load_data_robust(INPUT_CSV)
    if df_train is None or df_train.empty:
        raise ValueError("Cannot load training dataset or dataset is empty. Check INPUT_CSV!")

    ic50_candidates = [c for c in df_train.columns if 'ic50' in c.lower() and 'pic50' not in c.lower()]
    if not ic50_candidates:
        raise ValueError("No IC50 column found in training data.")
    ic50_col = ic50_candidates[0]
    smi_col_train = next((c for c in df_train.columns if 'smile' in c.lower()), 'Smiles')

    # Parse quantitative vs. right-censored IC50 measurements (verified in nM)
    parsed_ic50 = parse_ic50_column(df_train[ic50_col], cap_limit=IC50_CAP)
    n_total_raw = len(df_train)
    n_invalid = int((~parsed_ic50['is_valid_ic50']).sum())
    df_train = pd.concat([df_train, parsed_ic50], axis=1)
    df_train = df_train[df_train['is_valid_ic50']].copy()

    n_censored = int(df_train['is_right_censored'].sum())
    n_quant = len(df_train) - n_censored
    print(f">>> IC50 Parsing Summary (assumed unit: nM): Total={n_total_raw} | Quantitative={n_quant} | "
          f"Right-censored (> or >{IC50_CAP} nM)={n_censored} | Invalid/Missing={n_invalid}")

    if CENSORED_POLICY == "exclude" and n_censored > 0:
        print(f">>> Policy 'exclude': Removing {n_censored} right-censored observations.")
        df_train = df_train[~df_train['is_right_censored']].copy()
        df_train['capped_ic50'] = df_train['raw_ic50_numeric']
    else:
        print(f">>> Policy 'cap': Substituting {n_censored} right-censored observations at detection limit ({IC50_CAP} nM).")
        df_train['capped_ic50'] = df_train['raw_ic50_numeric'].clip(upper=IC50_CAP)

    df_train['pIC50'] = 9.0 - np.log10(df_train['capped_ic50'])

    # Standardize SMILES and log charge-neutralization changes
    std_results = df_train[smi_col_train].apply(lambda s: standardize_smiles(s, return_uncharge_flag=True))
    df_train['std_smiles'] = [r[0] for r in std_results]
    n_uncharged = int(sum(r[1] for r in std_results))
    print(f">>> Structure Standardization: {n_uncharged} structures modified by charge neutralization (Uncharger).")

    df_train = df_train.dropna(subset=['std_smiles', 'pIC50']).reset_index(drop=True)

    if DEDUPLICATE_TRAIN_SMILES:
        df_train = summarize_and_deduplicate_smiles(df_train, smi_col_train)

    X_train, y_train, desc_names, valid_train_idx, train_fps = compute_features_inference(df_train)
    train_smiles = df_train.loc[valid_train_idx, 'std_smiles'].values
    n_desc = len(desc_names)

    # Phase 1: 80/20 Hold-Out External Test Validation
    holdout_results = None
    if RUN_HOLDOUT_EXTERNAL_TEST:
        holdout_results = run_holdout_external_validation(
            X_train, y_train, n_desc, train_fps, train_smiles,
            split_ratio=TEST_SPLIT_RATIO, mode=HOLDOUT_SPLIT_MODE, fold_index=HOLDOUT_FOLD_INDEX
        )

    # Phase 2: Full-Dataset Cross-Validation & Production Model Refit
    model_bundle = train_final_stacking(X_train, y_train, n_desc, train_fps, train_smiles)
    model_bundle['desc_names'] = desc_names
    model_bundle['holdout_validation'] = holdout_results

    model_path = OUT_DIR / "Stacking_SVR_Final.pkl"
    joblib.dump(model_bundle, model_path)
    print(f"\n Saved production model bundle to: {model_path}")

    # Phase 3: Virtual Screening / Prediction on New Compounds (PT0x & Pyr)
    print("\n" + "="*68)
    print("  PHASE 3: PREDICTING NEW COMPOUNDS (DAXY) - EXACT INPUT ORDER")
    print("="*68)

    if not os.path.exists(PREDICT_CSV):
        raise FileNotFoundError(f"Cannot find prediction file at: {PREDICT_CSV}!")

    df_new = load_data_robust(PREDICT_CSV)
    if df_new is None or df_new.empty:
        raise ValueError("Prediction dataset could not be loaded or is empty.")

    smi_col_new = next((c for c in df_new.columns if 'smile' in c.lower()), None)
    if not smi_col_new:
        raise ValueError("Cannot find SMILES column in prediction dataset!")

    print(f">>> Found {len(df_new)} rows in {PREDICT_CSV}")
    df_out = df_new.copy()
    df_out['Original_Row_ID'] = np.arange(1, len(df_out) + 1)
    df_out['std_smiles'] = df_out[smi_col_new].apply(standardize_smiles)

    X_new, valid_new_idx, new_fps = compute_features_inference(df_out, training_desc_names=desc_names)

    print("Running ensemble predictions!")
    p1 = model_bundle['xgb'].predict(X_new)
    p2 = model_bundle['rf'].predict(X_new)
    p3 = model_bundle['et'].predict(X_new)
    p4 = model_bundle['svr'].predict(X_new)

    X_meta_new = np.column_stack([p1, p2, p3, p4])
    final_pred_pIC50 = model_bundle['meta'].predict(X_meta_new)
    pred_std = np.std(X_meta_new, axis=1, ddof=1)
    final_pred_IC50_nM = 10 ** (9.0 - final_pred_pIC50)

    ad_results = evaluate_new_applicability_domain(X_new, new_fps, model_bundle['ad_bundle'])

    # Assign predictions according to valid_new_idx:
    df_out.loc[valid_new_idx, 'Pred_XGB'] = p1
    df_out.loc[valid_new_idx, 'Pred_RF'] = p2
    df_out.loc[valid_new_idx, 'Pred_ET'] = p3
    df_out.loc[valid_new_idx, 'Pred_SVR'] = p4
    df_out.loc[valid_new_idx, 'Predicted_pIC50'] = final_pred_pIC50
    df_out.loc[valid_new_idx, 'BaseModel_Prediction_Disagreement'] = pred_std
    df_out.loc[valid_new_idx, 'Predicted_IC50_nM'] = final_pred_IC50_nM
    df_out.loc[valid_new_idx, 'Predicted_IC50_uM'] = final_pred_IC50_nM / 1000.0

    for k, v in ad_results.items():
        df_out.loc[valid_new_idx, k] = v

    # Save output file:
    out_file_main = OUT_DIR / "Daxy_Predictions_Stacking_with_AD.csv"
    df_out.to_csv(out_file_main, index=False, encoding='utf-8-sig')

    print(f"\n Done. Saved at: {out_file_main}")
    cols_show = ['Original_Row_ID', smi_col_new, 'Predicted_pIC50', 'Predicted_IC50_uM', 'BaseModel_Prediction_Disagreement', 'AD_Reliability']
    print("\nFirst 10 rows of predictions:")
    print(df_out[cols_show].head(10).to_string(index=False))

if __name__ == "__main__":
    main()